# The transfer flow: all the designs

Runs all the computations of `DESIGN-Transfer.md` and compares them. For every seed and every target robot, the flow trains or uses the following encoders, then fits the regressors on K labeled frames of the target and evaluates them on its `vp_testing` demonstrations:
- **baseline:** the encoder trained on the target (`native`), the encoders trained on each other robot (`single-...-from-<source>`), and the random-projection references (`randproj-...`). The trivial `mean` baseline is computed in every regressor stage.
- **D1:** an encoder trained on all the other robots, with one head each (`multi-...`).
- **D2:** the spatial keypoint latent (`kp-...`).
- **D3:** robot masking (`mask-from-...`), and background swap with color jitter (`maskswap-from-...`).
- **D4a:** BatchNorm re-estimated on unlabeled target frames, for the ViT and the Conv-VAE (`adabn-...`).
- **D4b:** Conv-VAEs trained on the target (`vae-native`, on `sp_training`; `vae-target`, on the regressor's own frames) and on each source (`vae-from-...`).
- **D5:** meta-learning with a ridge head (`meta-ridge`), and its first-order MAML variant (`meta-fomaml`).

The final results are in `visual_proprioception_transfer_compare/<compare_run>`: `summary.csv`, `summary_joints.csv` and `comparison_<target>.png`.

**Cost.** With the defaults (4 robots, 2 backbones, 3 seeds), the flow trains about 200 sensor processors and runs about 450 regressor stages. It needs a CUDA machine. On szenes, run `Flow_Transfer_Smoke.ipynb` instead.

In [ ]:
import sys
sys.path.append("../..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
from flow import setup_flow, run_flow, display_flow_report
from visual_proprioception.transfer.transfer_flow import DESIGNS, FAMILIES, TransferFlow, list_available_packs

In [ ]:
flow_name = "Transfer_flow_00"
# the datasets of this machine: data/expruns/transfer_datasets/<datasets_run>.yaml
datasets_run = "szenes"
# the short names of the packs (robots) of the flow; each robot is a source for the others
packs = ["al5d", "panda", "g1", "hand"]
# the robots to transfer to
targets = ["al5d", "panda", "g1", "hand"]
# the backbones of the baseline and of D1; D2 and D3 always use VGG19
backbones = ["vgg19", "vit_b_16"]
designs = DESIGNS
latent_size = 128
seeds = [0, 1, 2]
# the numbers of labeled target frames the regressors are fitted on; None: all of them
k_shots = [5, 10, 20, 50, 100, 200, None]
# use every frame_step-th frame of the demonstrations
frame_step = 1
epochs_sp = 100
epochs_vae = 100
meta_steps = 20000
regressor_steps = 3000
regressors = ["mlp", "ridge"]
# the group of the target the regressors are evaluated on
eval_group = "vp_testing"
creation_style = "exist-ok"

In [ ]:
# the packs available on this machine (see DESIGN-Transfer.md, "Datasets on other machines")
list_available_packs()

expruns_path, results_path, notebooks_path = setup_flow(flow_name, FAMILIES)
builder = TransferFlow({
    "flow_name": flow_name, "datasets_run": datasets_run, "packs": packs, "targets": targets,
    "backbones": backbones, "designs": designs, "latent_size": latent_size, "seeds": seeds,
    "k_shots": k_shots, "frame_step": frame_step, "epochs_sp": epochs_sp, "epochs_vae": epochs_vae,
    "meta_steps": meta_steps, "regressor_steps": regressor_steps, "regressors": regressors,
    "eval_group": eval_group, "creation_style": creation_style})
builder.import_packs()
expruns, compare_run = builder.build()
print(f"{len(expruns)} stages, of which {len(builder.vp_runs)} regressor stages")

In [ ]:
flow_error = None
try:
    run_flow(expruns, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
final_results_path = pathlib.Path(results_path, "visual_proprioception_transfer_compare", compare_run)
report = display_flow_report(
    expruns, results_path, notebooks_path, final_results_path,
    flow_error=flow_error,
    preview_files=sorted(final_results_path.glob("comparison_*.png")))
if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")